# Sentinel Orchestrator Test

Test the Sentinel-integrated MAF orchestrator with real agents and 3-thread supervision.

In [1]:
# Import Sentinel Orchestrator
from global_.sentinel_orchestrator import SentinelOrchestrator
from global_.state import Orchestrator_Global_State
from MAFI.agent_factory.graph_factory_concurrent import SubgraphFactory

In [2]:
# Create Sentinel Orchestrator with 3-thread supervision
orc = SentinelOrchestrator(
    auto_load_skills=True,
    db_path="subgraph_registry.db",
    SubgraphFactory=SubgraphFactory,
    quality_threshold=0.75,
    enable_supervision=True,  # Enable 3-thread supervision
).build()

✓ Built orchestrator with supervised agents


In [3]:
# Create initial state with user query
initial_state = Orchestrator_Global_State(
    user_id="test_123",
    user_query="hi"
)

In [4]:
# Execute with 3-thread supervision
# Actor thread: Executes the agent
# Observer thread: Watches runtime_memory
# Governor thread: Can intervene with instructions

result = await orc.ainvoke(initial_state)

In [5]:
# View the final answer
result['final_answer']

['Agent returned no result']

In [6]:
# View supervision metadata
print("Supervision Enabled:", result['supervision_enabled'])
print("\nSupervision Metadata:")
print(result['supervision_metadata'])

Supervision Enabled: True

Supervision Metadata:
{'observer_observations': [{'timestamp': 1784626651.2863932, 'status': 'executing', 'memory_snapshot': None, 'latency': None}, {'timestamp': 1784626651.339572, 'status': 'error', 'memory_snapshot': None, 'latency': None}], 'governor_interventions': [{'type': 'retry', 'reason': 'actor_error'}]}


In [7]:
# View Observer observations
if result['supervision_metadata']:
    observations = result['supervision_metadata']['observer_observations']
    print(f"Observer tracked {len(observations)} state changes:")
    for i, obs in enumerate(observations, 1):
        print(f"  {i}. Status: {obs['status']}")

Observer tracked 2 state changes:
  1. Status: executing
  2. Status: error


In [8]:
# View Governor interventions
if result['supervision_metadata']:
    interventions = result['supervision_metadata']['governor_interventions']
    print(f"Governor made {len(interventions)} interventions:")
    for i, intervention in enumerate(interventions, 1):
        print(f"  {i}. Type: {intervention['type']}")
        if 'quality' in intervention:
            print(f"     Quality: {intervention['quality']:.2f}")

Governor made 1 interventions:
  1. Type: retry


In [9]:
# View orchestrator statistics
print("Orchestrator Statistics:")
stats = result['orchestrator_stats']
for key, value in stats.items():
    print(f"  {key}: {value}")

Orchestrator Statistics:
  total_executions: 1
  total_observations: 2
  total_interventions: 1


In [10]:
# Get metrics from orchestrator
metrics = orc.get_metrics()
print("\nOrchestrator Metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")


Orchestrator Metrics:
  execution_count: 1
  total_observations: 2
  total_interventions: 1
  supervision_enabled: True
  agents: ['intent']


## Test with Different Query

In [11]:
# Test with a more complex query
complex_state = Orchestrator_Global_State(
    user_id="test_456",
    user_query="How do I build a secure authentication system for my web application?"
)

result2 = await orc.ainvoke(complex_state)

In [12]:
# View intent understanding output
result2['intent_understanding']

{'intent_understanding': 'Agent returned no result'}

In [13]:
# View supervision for this execution
if result2['supervision_metadata']:
    print(f"Observations: {len(result2['supervision_metadata']['observer_observations'])}")
    print(f"Interventions: {len(result2['supervision_metadata']['governor_interventions'])}")

Observations: 2
Interventions: 1


## Compare: Supervision ON vs OFF

In [14]:
# Create orchestrator WITHOUT supervision
orc_no_supervision = SentinelOrchestrator(
    auto_load_skills=True,
    db_path="subgraph_registry.db",
    SubgraphFactory=SubgraphFactory,
    enable_supervision=False,  # Disable supervision
).build()

result_no_sup = await orc_no_supervision.ainvoke(initial_state)

✓ Built orchestrator with standard agents


AttributeError: 'set' object has no attribute 'get'

In [15]:
# Compare results
print("WITH Supervision:")
print(f"  Supervision enabled: {result['supervision_enabled']}")
print(f"  Metadata available: {result['supervision_metadata'] is not None}")
print()
print("WITHOUT Supervision:")
print(f"  Supervision enabled: {result_no_sup['supervision_enabled']}")
print(f"  Metadata available: {result_no_sup['supervision_metadata'] is not None}")

WITH Supervision:
  Supervision enabled: True
  Metadata available: True

WITHOUT Supervision:


NameError: name 'result_no_sup' is not defined

## 3-Thread Architecture Visualization

```
┌────────────────────────────────────┐
│   SentinelExecutionState (Shared)  │
│   • runtime_memory                 │
│   • runtime_instructions           │
└────────────────────────────────────┘
     │            │            │
Thread 1      Thread 2     Thread 3
     │            │            │
     ▼            ▼            ▼
┌─────────┐  ┌─────────┐  ┌─────────┐
│  ACTOR  │  │OBSERVER │  │GOVERNOR │
│         │  │         │  │         │
│ Execute │  │ Watch   │  │ Monitor │
│ agent   │  │ memory  │  │ quality │
│         │  │         │  │         │
│ Update  │  │ Log     │  │ Issue   │
│ memory  │  │ changes │  │ retries │
└─────────┘  └─────────┘  └─────────┘
```